In [1]:
# Cell 1 — Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print("Pandas version:", pd.__version__)

Libraries imported successfully
Pandas version: 2.3.3


In [2]:
# Cell 2 — Load Data

df = pd.read_csv(r'C:\weather-ai-project\data\weather_eda.csv')

print("Shape:", df.shape)
print("Columns:", list(df.columns))

Shape: (145460, 26)
Columns: ['Date', 'Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'RainTomorrow', 'Year', 'Month', 'Season']


In [3]:
# Cell 3 — Check Missing Values

missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})

missing = missing[missing['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(missing)

               Missing Count  Missing %
Sunshine               69835      48.01
Evaporation            62790      43.17
Cloud3pm               59358      40.81
Cloud9am               55888      38.42
Pressure9am            15065      10.36
Pressure3pm            15028      10.33
WindDir9am             10566       7.26
WindGustDir            10326       7.10
WindGustSpeed          10263       7.06
Humidity3pm             4507       3.10
WindDir3pm              4228       2.91
Temp3pm                 3609       2.48
RainTomorrow            3267       2.25
Rainfall                3261       2.24
RainToday               3261       2.24
WindSpeed3pm            3062       2.11
Humidity9am             2654       1.82
Temp9am                 1767       1.21
WindSpeed9am            1767       1.21
MinTemp                 1485       1.02
MaxTemp                 1261       0.87


In [4]:
# Cell 4 — Drop Null Target Rows

print("Before drop:", df.shape)
df.dropna(subset=['RainTomorrow'], inplace=True)
print("After drop:", df.shape)
print("RainTomorrow nulls:", df['RainTomorrow'].isnull().sum())

Before drop: (145460, 26)
After drop: (142193, 26)
RainTomorrow nulls: 0


In [6]:
# Cell 5 — Fill Numerical Columns with Median

num_cols = df.select_dtypes(include=np.number).columns.tolist()
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)
print("Numerical columns filled:", num_cols)
print("Numerical nulls remaining:", df[num_cols].isnull().sum().sum())

Numerical columns filled: ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'Year', 'Month']
Numerical nulls remaining: 0


In [7]:
# Cell 6 — Fill Categorical Columns with Mode

cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("Categorical columns filled:", cat_cols)
print("Categorical nulls remaining:", df[cat_cols].isnull().sum().sum())
print("Total nulls in dataset:", df.isnull().sum().sum())

Categorical columns filled: ['Date', 'Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday', 'RainTomorrow', 'Season']
Categorical nulls remaining: 0
Total nulls in dataset: 0


In [8]:
# Cell 7 — Encode Categorical Columns

le = LabelEncoder()

encode_cols = ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm',
               'RainToday', 'RainTomorrow', 'Season']

for col in encode_cols:
    df[col] = le.fit_transform(df[col])

print("Columns encoded:", encode_cols)
print(df[encode_cols].head(3))

Columns encoded: ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday', 'RainTomorrow', 'Season']
   Location  WindGustDir  WindDir9am  WindDir3pm  RainToday  RainTomorrow  \
0         2           13          13          14          0             0   
1         2           14           6          15          0             0   
2         2           15          13          15          0             0   

   Season  
0       2  
1       2  
2       2  


In [9]:
# Cell 8 — Outlier Detection

num_cols = df.select_dtypes(include=np.number).columns.tolist()
outlier_summary = {}

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)].shape[0]
    outlier_summary[col] = outliers

outlier_df = pd.Series(outlier_summary).sort_values(ascending=False)
print(outlier_df)

Sunshine         60823
Evaporation      32317
RainTomorrow     31877
RainToday        31455
Rainfall         28545
WindGustSpeed     5386
Cloud3pm          4957
Pressure9am       2708
WindSpeed3pm      2458
Pressure3pm       2305
WindSpeed9am      1739
Humidity9am       1419
Temp3pm            840
MaxTemp            459
Temp9am            292
MinTemp             62
Month                0
Year                 0
Location             0
Cloud9am             0
Humidity3pm          0
WindDir3pm           0
WindDir9am           0
WindGustDir          0
Season               0
dtype: int64


In [10]:
# Cell 9 — Cap Outliers

skip_cols = ['RainToday', 'RainTomorrow', 'Location', 'WindGustDir',
             'WindDir9am', 'WindDir3pm', 'Season', 'Year', 'Month']

cap_cols = [col for col in df.select_dtypes(include=np.number).columns if col not in skip_cols]

for col in cap_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

print("Outliers capped for:", cap_cols)
print("Done")

Outliers capped for: ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm']
Done


In [11]:
# Cell 10 — Feature Scaling

scale_cols = ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine',
              'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am',
              'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am',
              'Cloud3pm', 'Temp9am', 'Temp3pm']

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("Scaling done for:", scale_cols)
print(df[scale_cols].describe().round(2))

Scaling done for: ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm']
         MinTemp    MaxTemp   Rainfall  Evaporation   Sunshine  WindGustSpeed  \
count  142193.00  142193.00  142193.00    142193.00  142193.00      142193.00   
mean        0.00      -0.00       0.00         0.00      -0.00          -0.00   
std         1.00       1.00       1.00         1.00       1.00           1.00   
min        -2.88      -2.93      -0.63        -1.78      -1.51          -2.55   
25%        -0.72      -0.75      -0.63        -0.49      -0.40          -0.71   
50%        -0.03      -0.09      -0.63         0.00       0.04          -0.05   
75%         0.72       0.70       0.35         0.37       0.34           0.52   
max         2.88       2.88       1.83         1.66       1.44           2.37   

       WindSpeed9am  WindSpeed3pm  

In [12]:
# Cell 11 — Feature Engineering

df['Temp_Range'] = df['MaxTemp'] - df['MinTemp']
df['Humidity_Drop'] = df['Humidity9am'] - df['Humidity3pm']
df['Pressure_Drop'] = df['Pressure9am'] - df['Pressure3pm']
df['Wind_Change'] = df['WindSpeed3pm'] - df['WindSpeed9am']

print("New features added:")
print("Temp_Range, Humidity_Drop, Pressure_Drop, Wind_Change")
print("\nShape after feature engineering:", df.shape)
print(df[['Temp_Range', 'Humidity_Drop', 'Pressure_Drop', 'Wind_Change']].head(3))

New features added:
Temp_Range, Humidity_Drop, Pressure_Drop, Wind_Change

Shape after feature engineering: (142193, 30)
   Temp_Range  Humidity_Drop  Pressure_Drop  Wind_Change
0   -0.236406       1.547804      -0.268393    -0.068670
1    1.013067      -0.036083       0.065196     1.565600
2    0.236675      -0.599109      -0.527686     0.285034


In [13]:
# Cell 12 — Drop Date Column

df.drop(columns=['Date'], inplace=True)

print("Date column dropped")
print("Final shape:", df.shape)
print("Columns:", list(df.columns))

Date column dropped
Final shape: (142193, 29)
Columns: ['Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'RainTomorrow', 'Year', 'Month', 'Season', 'Temp_Range', 'Humidity_Drop', 'Pressure_Drop', 'Wind_Change']


In [14]:
# Cell 13 — Class Imbalance Check

rain_counts = df['RainTomorrow'].value_counts()
rain_percent = df['RainTomorrow'].value_counts(normalize=True) * 100

print("RainTomorrow Value Counts:")
print(rain_counts)
print("\nRainTomorrow Percentage:")
print(rain_percent.round(2))

ratio = rain_counts[0] / rain_counts[1]
print(f"\nImbalance Ratio: {ratio:.2f}:1")
print("Note: Class imbalance exists — handle with class_weight in Notebook 4")

RainTomorrow Value Counts:
RainTomorrow
0    110316
1     31877
Name: count, dtype: int64

RainTomorrow Percentage:
RainTomorrow
0    77.58
1    22.42
Name: proportion, dtype: float64

Imbalance Ratio: 3.46:1
Note: Class imbalance exists — handle with class_weight in Notebook 4


In [15]:
# Cell 14 — Save Scaler with Joblib

import joblib

joblib.dump(scaler, r'C:\weather-ai-project\models\scaler.pkl')

print("Scaler saved: C:\\weather-ai-project\\models\\scaler.pkl")
print("Scaler type:", type(scaler))
print("Scaled columns count:", len(scale_cols))
print("Use this scaler in web app to scale new input before prediction")

Scaler saved: C:\weather-ai-project\models\scaler.pkl
Scaler type: <class 'sklearn.preprocessing._data.StandardScaler'>
Scaled columns count: 16
Use this scaler in web app to scale new input before prediction


In [16]:
# Cell 15 — Final Data Verification

print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nTotal Nulls:", df.isnull().sum().sum())
print("\nSample Row:")
print(df.head(2))

Shape: (142193, 29)

Data Types:
Location           int64
MinTemp          float64
MaxTemp          float64
Rainfall         float64
Evaporation      float64
Sunshine         float64
WindGustDir        int64
WindGustSpeed    float64
WindDir9am         int64
WindDir3pm         int64
WindSpeed9am     float64
WindSpeed3pm     float64
Humidity9am      float64
Humidity3pm      float64
Pressure9am      float64
Pressure3pm      float64
Cloud9am         float64
Cloud3pm         float64
Temp9am          float64
Temp3pm          float64
RainToday          int64
RainTomorrow       int64
Year               int64
Month              int64
Season             int64
Temp_Range       float64
Humidity_Drop    float64
Pressure_Drop    float64
Wind_Change      float64
dtype: object

Total Nulls: 0

Sample Row:
   Location   MinTemp   MaxTemp  Rainfall  Evaporation  Sunshine  WindGustDir  \
0         2  0.190084 -0.046323  0.351559     0.002906  0.040602           13   
1         2 -0.749182  0.263885 -0.63

In [17]:
# Cell 16 — Save Cleaned Data

df.to_csv(r'C:\weather-ai-project\data\weather_preprocessed.csv', index=False)

print("File saved: weather_preprocessed.csv")
print("Location: C:\\weather-ai-project\\data\\")
print("Final shape:", df.shape)
print("Ready for Notebook 4 — Model Training")

File saved: weather_preprocessed.csv
Location: C:\weather-ai-project\data\
Final shape: (142193, 29)
Ready for Notebook 4 — Model Training
